In [1]:
import numpy as np
import os
import sys
import subprocess
import time 

# Navigate to the parent directory of the project structure
project_dir = os.path.abspath(os.path.join(os.getcwd(), '../..'))
src_dir = os.path.join(project_dir, 'src')
log_dir = os.path.join(project_dir, 'log')
fig_dir = os.path.join(project_dir, 'fig')
data_dir = os.path.join(project_dir, 'build')
scripts_dir = os.path.join(project_dir, 'scripts')
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

# Add the src directory to sys.path
sys.path.append(src_dir)


from boolean_circuit.graph_synthesizer import CircuitGraph
from boolean_circuit.partitioner import KaHIPPartitioner

In [2]:
def parse_stats_file(path: str):
    """Return (client_comp, client_comm, baseline_comp, baseline_comm) from a stats file."""
    with open(path, "r") as f:
        lines = f.readlines()

    client_comp = None
    client_comm = None
    baseline_comp = None
    baseline_comm = None

    for line in lines:
        if "max_client_balance_computation_cost (s)" in line:
            client_comp = float(line.split(":", 1)[1])
        elif "max_client_balance_communication_cost (MB)" in line:
            client_comm = float(line.split(":", 1)[1])
        elif "baseline computation cost (s)" in line:
            baseline_comp = float(line.split(":", 1)[1])
        elif "baseline communication cost (MB)" in line:
            baseline_comm = float(line.split(":", 1)[1])

    return client_comp, client_comm, baseline_comp, baseline_comm

In [3]:
# Set the number of parts/elements for the oblivious sort
NPARTS = 32  # change this value as needed
circuit_name = "oblivious_sort"

script = os.path.join(scripts_dir, "boolean_circuit", f"{circuit_name}_cost_estimate.sh")

start_time = time.time()
result = subprocess.run(
    [script, "-p", str(NPARTS)],
    cwd=project_dir,
    capture_output=True,
    text=True,
    check=False,
)
end_time = time.time()

print(f"Script execution time: {end_time - start_time} seconds")

if result.returncode != 0:
    print("Return code:", result.returncode)
    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)
else:
    # Read statistics from the generated stats file
    stats_dir = os.path.join(
        data_dir,
        "boolean_circuits",
        f"{circuit_name}_u32_N{NPARTS}",
    )
    stats_file = os.path.join(stats_dir, "oblivious_sorting_stats.txt")

    if not os.path.exists(stats_file):
        print(f"Stats file not found: {stats_file}")
    else:
        client_comp, client_comm, baseline_comp, baseline_comm = parse_stats_file(stats_file)

        print("Parsed stats:")
        print("  client computation (s):", client_comp)
        print("  client bandwidth (MB):", client_comm)
        print("  baseline computation (s):", baseline_comp)
        print("  baseline bandwidth (MB):", baseline_comm)

Script execution time: 2.2544775009155273 seconds
Parsed stats:
  client computation (s): 12.99
  client bandwidth (MB): 0.507063
  baseline computation (s): 10.24005
  baseline bandwidth (MB): 0.758784
